In [1]:
import pandas as pd
import numpy as np

csv_path = r'C:\Users\Vido\Desktop\EPL448\accepted_2007_to_2018q4.csv\accepted_2007_to_2018q4.csv'
preview_rows = 50000

df = pd.read_csv(csv_path, nrows=preview_rows, low_memory=True)



C:\Users\Vido\AppData\Local\Temp\ipykernel_140204\1502689034.py:7: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, nrows=preview_rows, low_memory=True)


In [2]:
# 1) Understand dataset structure
print('Shape:', df.shape)
display(df.head(5))
display(pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str)}).head(30))
df.info()


Shape: (50000, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


,column,dtype
id,id,int64
member_id,member_id,float64
loan_amnt,loan_amnt,float64
funded_amnt,funded_amnt,float64
funded_amnt_inv,funded_amnt_inv,float64
term,term,object
int_rate,int_rate,float64
installment,installment,float64
grade,grade,object
sub_grade,sub_grade,object


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Columns: 151 entries, id to settlement_term
dtypes: float64(114), int64(1), object(36)
memory usage: 57.6+ MB


In [ ]:
# 2) Summary statistics
num_df = df.select_dtypes(include='number')
cat_df = df.select_dtypes(exclude='number')

print('Numeric columns:', num_df.shape[1])
if num_df.shape[1] > 0: 
    display(num_df.describe().T)

print('Categorical columns:', cat_df.shape[1])
if cat_df.shape[1] > 0:
    cat_summary = pd.DataFrame({
        'unique_values': cat_df.nunique(dropna=True), # to undestand what level categoric variable we are working with
        'missing_values': cat_df.isna().sum() #calculate sum missing values
    }).sort_values('unique_values', ascending=False)
    display(cat_summary.head(20))


Numeric columns: 115


,count,mean,std,min,25%,50%,75%,max
id,50000.0,6.670963e+07,1.942339e+06,67025.00,6.594684e+07,6.654286e+07,6.744762e+07,68617057.00
member_id,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_amnt,50000.0,1.501936e+04,8.676103e+03,1000.00,8.000000e+03,1.400000e+04,2.000000e+04,35000.00
funded_amnt,50000.0,1.501936e+04,8.676103e+03,1000.00,8.000000e+03,1.400000e+04,2.000000e+04,35000.00
funded_amnt_inv,50000.0,1.501182e+04,8.671132e+03,950.00,8.000000e+03,1.390000e+04,2.000000e+04,35000.00
...,...,...,...,...,...,...,...,...
hardship_payoff_balance_amount,406.0,1.043131e+04,6.537326e+03,55.73,5.190170e+03,9.388780e+03,1.505548e+04,29401.04
hardship_last_payment_amount,406.0,1.914970e+02,1.871277e+02,0.05,4.635500e+01,1.305150e+02,2.846375e+02,926.41
settlement_amount,1537.0,5.009781e+03,3.567857e+03,186.00,2.110000e+03,4.292650e+03,7.024000e+03,22000.00
settlement_percentage,1537.0,4.700015e+01,5.367391e+00,30.00,4.500000e+01,4.500000e+01,5.000000e+01,75.00


Categorical columns: 36


,unique_values,missing_values
url,50000,0
emp_title,21288,2995
zip_code,856,0
earliest_cr_line,605,0
addr_state,49,0
last_credit_pull_d,41,1
last_pymnt_d,40,34
sub_grade,35,0
settlement_date,35,48463
debt_settlement_flag_date,30,48463


In [ ]:
# 3) Detect anomalies / data quality issues
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
quality = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
display(quality[quality['missing_count'] > 0]) 

dup_count = df.duplicated().sum()
print('Duplicate rows:', int(dup_count))


,missing_count,missing_pct
member_id,50000,100.00
sec_app_collections_12_mths_ex_med,50000,100.00
sec_app_chargeoff_within_12_mths,50000,100.00
sec_app_num_rev_accts,50000,100.00
sec_app_open_act_il,50000,100.00
...,...,...
title,132,0.26
last_pymnt_d,34,0.07
revol_util,24,0.05
dti,1,0.00


Duplicate rows: 0


In [ ]:
# 4) Explore numeric patterns by loan status only
group_col = 'loan_status'
num_cols = df.select_dtypes(include='number').columns.tolist()

if group_col in df.columns and num_cols:
    print(f'Grouping by: {group_col}')
    # Rows are loan_status categories, columns are stats for each numeric feature
    grouped = (df[[group_col] + num_cols]
               .dropna(subset=[group_col])
               .groupby(group_col)[num_cols]
               .agg(['count', 'mean', 'median', 'min', 'max']))
    display(grouped)
else:
    print("'loan_status' column not found or no numeric columns exist.")


Grouping by: loan_status


id                                                \
                    count          mean      median       min       max   
loan_status                                                               
Charged Off          9027  6.670832e+07  66545850.0    727733  68616394   
Current              5610  6.665262e+07  66511674.0    614874  68617034   
Default                 1  6.651050e+07  66510505.0  66510505  66510505   
Fully Paid          34978  6.671863e+07  66544636.0     67025  68617057   
In Grace Period       100  6.681430e+07  66599883.0  65098275  68615169   
Late (16-30 days)      38  6.649158e+07  66482915.5  64832258  68493489   
Late (31-120 days)    246  6.677068e+07  66602342.0  41051455  68605999   

                   member_id                      ... settlement_percentage  \
                       count mean median min max  ...                 count   
loan_status                                       ...                         
Charged Off                0  NaN    NaN NaN NaN  ...                  1520   
Current                    0  NaN    NaN NaN NaN  ...                     0   
Default                    0  NaN    NaN NaN NaN  ...                     0   
Fully Paid                 0  NaN    NaN NaN NaN  ...                     2   
In Grace Period            0  NaN    NaN NaN NaN  ...                     0   
Late (16-30 days)          0  NaN    NaN NaN NaN  ...                     0   
Late (31-120 days)         0  NaN    NaN NaN NaN  ...                    15   

                                                   settlement_term             \
                         mean median    min    max           count       mean   
loan_status                                                                     
Charged Off         47.012651   45.0  30.00  75.00            1520  13.698684   
Current                   NaN    NaN    NaN    NaN               0        NaN   
Default                   NaN    NaN    NaN    NaN               0        NaN   
Fully Paid          45.000000   45.0  45.00  45.00               2   6.000000   
In Grace Period           NaN    NaN    NaN    NaN               0        NaN   
Late (16-30 days)         NaN    NaN    NaN    NaN               0        NaN   
Late (31-120 days)  46.000000   45.0  44.99  50.01              15  16.533333   

                                       
                   median   min   max  
loan_status                            
Charged Off          14.0   0.0  65.0  
Current               NaN   NaN   NaN  
Default               NaN   NaN   NaN  
Fully Paid            6.0   4.0   8.0  
In Grace Period       NaN   NaN   NaN  
Late (16-30 days)     NaN   NaN   NaN  
Late (31-120 days)   18.0  12.0  18.0  

[7 rows x 575 columns]

In [6]:
# 5) Binning (numeric -> categorical groups) // Finding quantiles of loan amount 
num_cols = df.select_dtypes(include='number').columns.tolist()
if num_cols:
    bin_col = num_cols[2]
    q = df[bin_col].quantile([0.25, 0.5, 0.75]).values
    bins = [-np.inf, q[0], q[1], q[2], np.inf]
    labels = ['Q1-low', 'Q2-mid-low', 'Q3-mid-high', 'Q4-high']

    df[f'{bin_col}_bin'] = pd.cut(df[bin_col], bins=bins, labels=labels, include_lowest=True)
    display(df[f'{bin_col}_bin'].value_counts(dropna=False))
else:
    print('No numeric columns found for binning.')


loan_amnt_bin
Q2-mid-low     13087
Q1-low         12713
Q3-mid-high    12113
Q4-high        12087
Name: count, dtype: int64

In [ ]:
# 6) Iterative re-check after a small cleaning step
clean_df = df.dropna(axis=1, how='all').copy() #Remove columns that are completely empty  
print('Before clean:', df.shape, '| After clean:', clean_df.shape)
display(clean_df.describe(include='number').T.head(10)) 

#!We already have them in the previous step...



Before clean: (50000, 152) | After clean: (50000, 138)


,count,mean,std,min,25%,50%,75%,max
id,50000.0,6.670963e+07,1.942339e+06,67025.00,65946836.25,66542859.00,6.744762e+07,68617057.00
loan_amnt,50000.0,1.501936e+04,8.676103e+03,1000.00,8000.00,14000.00,2.000000e+04,35000.00
funded_amnt,50000.0,1.501936e+04,8.676103e+03,1000.00,8000.00,14000.00,2.000000e+04,35000.00
funded_amnt_inv,50000.0,1.501182e+04,8.671132e+03,950.00,8000.00,13900.00,2.000000e+04,35000.00
int_rate,50000.0,1.223478e+01,4.191338e+00,5.32,9.17,11.99,1.448000e+01,28.99
installment,50000.0,4.340988e+02,2.474940e+02,14.77,255.04,378.15,5.737025e+02,1354.66
annual_inc,50000.0,7.919317e+04,1.013212e+05,0.00,48000.00,66000.00,9.500000e+04,9000000.00
dti,49999.0,1.933882e+01,9.810057e+00,0.00,12.68,18.82,2.561500e+01,999.00
delinq_2yrs,50000.0,3.455000e-01,9.102013e-01,0.00,0.00,0.00,0.000000e+00,15.00
fico_range_low,50000.0,6.944479e+02,3.089584e+01,660.00,670.00,685.00,7.100000e+02,845.00
